In [1]:
import os

In [2]:
%pwd

'/Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS/research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'/Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS'

In [5]:
import pandas as pd

data=pd.read_csv("artifacts/data_ingestion/winequality-red.csv")
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [7]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [8]:
data.shape

(1599, 12)

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataValidationConfig:
    root_dir:Path
    STATUS_FILE:str
    unzip_data_dir:Path
    all_schema:dict

In [10]:
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import *

# Verify working directory
print(f"Current working directory: {os.getcwd()}")
print(f"CONFIG_FILE_PATH: {CONFIG_FILE_PATH}")
print(f"File exists: {os.path.exists(CONFIG_FILE_PATH)}")


Current working directory: /Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS
CONFIG_FILE_PATH: config/config.yaml
File exists: True


In [11]:
from pathlib import Path

# Use absolute path from current working directory
PROJECT_ROOT = Path.cwd()
CONFIG_FILE_PATH_ABS = PROJECT_ROOT / "config" / "config.yaml"
PARAMS_FILE_PATH_ABS = PROJECT_ROOT / "params.yaml"
SCHEMA_FILE_PATH_ABS = PROJECT_ROOT / "schema.yaml"

print(f"Config path: {CONFIG_FILE_PATH_ABS}")
print(f"Config exists: {CONFIG_FILE_PATH_ABS.exists()}")

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH_ABS,
        params_filepath = PARAMS_FILE_PATH_ABS,
        schema_filepath = SCHEMA_FILE_PATH_ABS):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema,
        )

        logger.info(f"Data validation config created with {len(schema)} columns")
        return data_validation_config


Config path: /Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS/config/config.yaml
Config exists: True


In [12]:
import os
from src.wine_quality_prediction import logger
import pandas as pd

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        """Validate that all required columns exist in the dataset"""
        try:
            validation_status = True
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)
            all_schema = set(self.config.all_schema.keys())

            # Check if all schema columns exist in data
            missing_cols = all_schema - set(all_cols)
            if missing_cols:
                validation_status = False
                logger.error(f"Missing columns in data: {missing_cols}")
                self._write_status(validation_status)
                return validation_status

            # Check if data has any extra columns not in schema
            extra_cols = set(all_cols) - all_schema
            if extra_cols:
                validation_status = False
                logger.error(f"Extra columns in data not in schema: {extra_cols}")
                self._write_status(validation_status)
                return validation_status

            logger.info("All columns validation passed")
            self._write_status(validation_status)
            return validation_status

        except Exception as e:
            logger.error(f"Column validation failed: {str(e)}")
            self._write_status(False)
            raise e

    def validate_data_types(self) -> bool:
        """Validate that data types match the schema"""
        try:
            validation_status = True
            data = pd.read_csv(self.config.unzip_data_dir)
            
            type_mapping = {
                'int': ['int64', 'int32', 'int'],
                'float': ['float64', 'float32', 'float'],
                'object': ['object', 'string'],
                'string': ['object', 'string']
            }

            for column, dtype in self.config.all_schema.items():
                col_dtype = str(data[column].dtype)
                expected_type = dtype.get('dtype') if isinstance(dtype, dict) else dtype
                
                allowed_types = type_mapping.get(str(expected_type), [str(expected_type)])
                
                if col_dtype not in allowed_types:
                    validation_status = False
                    logger.error(f"Type mismatch for column '{column}': expected {expected_type}, got {col_dtype}")
                else:
                    logger.info(f"Column '{column}' type validation passed: {col_dtype}")

            self._write_status(validation_status)
            return validation_status

        except Exception as e:
            logger.error(f"Type validation failed: {str(e)}")
            self._write_status(False)
            raise e

    def validate_missing_values(self) -> bool:
        """Validate missing values in the dataset"""
        try:
            validation_status = True
            data = pd.read_csv(self.config.unzip_data_dir)
            
            missing_counts = data.isnull().sum()
            if missing_counts.sum() > 0:
                logger.warning(f"Missing values found:\n{missing_counts[missing_counts > 0]}")
                validation_status = False
            else:
                logger.info("No missing values found")

            self._write_status(validation_status)
            return validation_status

        except Exception as e:
            logger.error(f"Missing values validation failed: {str(e)}")
            self._write_status(False)
            raise e

    def _write_status(self, status: bool):
        """Helper method to write validation status to file"""
        try:
            with open(self.config.STATUS_FILE, 'w') as f:
                f.write(f"Validation status: {status}\n")
            logger.info(f"Validation status written to {self.config.STATUS_FILE}")
        except Exception as e:
            logger.error(f"Failed to write status file: {str(e)}")
            raise e


In [14]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    
    # Run all validation checks
    print("=" * 50)
    print("Running Data Validation Checks...")
    print("=" * 50)
    
    # Check columns
    col_status = data_validation.validate_all_columns()
    print(f"Column Validation: {'PASSED' if col_status else 'FAILED'}")
    
    # Check data types
    type_status = data_validation.validate_data_types()
    print(f"Type Validation: {'PASSED' if type_status else 'FAILED'}")
    
    # Check missing values
    missing_status = data_validation.validate_missing_values()
    print(f"Missing Values Validation: {'PASSED' if missing_status else 'FAILED'}")
    
    print("=" * 50)
    overall_status = col_status and type_status
    print(f"Overall Validation Status: {'PASSED ✓' if overall_status else 'FAILED ✗'}")
    print("=" * 50)
    
except Exception as e:
    logger.error(f"Validation pipeline failed: {str(e)}")
    raise e


[2026-04-16 21:58:29,868: INFO: common: yaml file: /Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS/config/config.yaml loaded successfully]
[2026-04-16 21:58:29,869: INFO: common: yaml file: /Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS/params.yaml loaded successfully]
[2026-04-16 21:58:29,876: INFO: common: yaml file: /Users/kishorkumarparoi/Desktop/MLOps/Project/Wine-Quality-DS/schema.yaml loaded successfully]
[2026-04-16 21:58:29,877: INFO: common: created directory at: artifacts]
[2026-04-16 21:58:29,878: INFO: common: created directory at: artifacts/data_validation]
[2026-04-16 21:58:29,879: INFO: 1733525586: Data validation config created with 12 columns]
Running Data Validation Checks...
[2026-04-16 21:58:29,883: INFO: 1116871581: All columns validation passed]
[2026-04-16 21:58:29,884: INFO: 1116871581: Validation status written to artifacts/data_validation/status.txt]
Column Validation: PASSED
[2026-04-16 21:58:29,887: INFO: 1116871581: Column 'fix